In [1]:
import os
print(os.getcwd())
os.chdir('/home/fatemeh/thesis/kinodata-3D-affinity-prediction')
print(os.getcwd())

/home/fatemeh/thesis/kinodata-3D-affinity-prediction/examples
/home/fatemeh/thesis/kinodata-3D-affinity-prediction


In [2]:
# import kinodata-3d-affinity-prediction
import kinodata
from kinodata.data import KinodataDocked
from kinodata.transform import TransformToComplexGraph
from kinodata.types import *


import json
from pathlib import Path
from typing import Any
from functools import partial

import torch
from torch_geometric.loader import DataLoader

import kinodata.configuration as cfg
# from kinodata.model import ComplexTransformer, DTIModel, RegressionModel
from kinodata.model import ComplexTransformer, RegressionModel
from kinodata.model.complex_transformer import make_model as make_complex_transformer
# from kinodata.model.dti import make_model as make_dti_baseline
from kinodata.data.data_module import make_kinodata_module
from kinodata.transform import TransformToComplexGraph

import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
import tqdm


!wandb disabled

W&B disabled.


## Whole Data
Saving a portion

In [3]:
data = KinodataDocked(transform=TransformToComplexGraph(remove_heterogeneous_representation=False))

In [ ]:
mini_data = data[:100]
mini_data
torch.save(mini_data, "data/probing/100subset_data.pt")

NameError: name 'data' is not defined

In [14]:
a_complex = mini_data[0]
torch.save(a_complex, "data/probing/a_complex.pt")

In [12]:
with open("data/probing/ident_to_idx.json", "w") as f:
    json.dump(mini_data.ident_index_map(), f)

### Looking at data

In [4]:
# print(data)
# print(dir(data))
print(data.__dict__)
# print(data[0])

{'remove_hydrogen': True, '_prefix': None, 'residue_representation': 'sequence', 'require_kissim_residues': False, 'use_multiprocessing': True, 'make_pyg_data': functools.partial(<function process_pyg at 0x7fed1d478a40>, residue_representation='sequence', require_kissim_residues=False), 'num_processes': 16, 'post_filter': None, 'root': '/home/fatemeh/thesis/kinodata-3D-affinity-prediction/data', 'transform': TransformToComplexGraph(), 'pre_transform': None, 'pre_filter': FilterActivityType(allowed=pIC50), 'log': True, '_indices': None, 'force_reload': False, '_data': HeteroData(
  y=[41238],
  docking_score=[41238],
  posit_prob=[41238],
  predicted_rmsd=[41238],
  pocket_sequence=[41238],
  scaffold=[41238],
  activity_type=[41238],
  ident=[41238],
  smiles=[41238],
  ligand={
    z=[1288419],
    x=[1288419, 12],
    pos=[1288419, 3],
  },
  pocket={
    z=[27393111],
    x=[27393111, 12],
    pos=[27393111, 3],
  },
  pocket_residue={ x=[3498751, 23] },
  (ligand, bond, ligand)={
 

In [ ]:
# print(data.{'pocket'})
# print(data.data.keys())
# print(data.data.edge_types)
# print(data.data.node_types)
# print(data[2].edge_attrs)
# print(data[2].node_attrs)
# print(data[2].edge_stores)
print(data[2])

['posit_prob', 'pos', 'scaffold', 'ident', 'y', 'edge_attr', 'pocket_sequence', 'activity_type', 'edge_index', 'docking_score', 'predicted_rmsd', 'z', 'x', 'smiles']


In [ ]:
data.data

/home/fatemeh/miniconda3/envs/kinodata3d/lib/python3.12/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. The data of the dataset is already cached, so any modifications to `data` will not be reflected when accessing its elements. Clearing the cache now by removing all elements in `dataset._data_list`. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


HeteroData(
  y=[41238],
  docking_score=[41238],
  posit_prob=[41238],
  predicted_rmsd=[41238],
  pocket_sequence=[41238],
  scaffold=[41238],
  activity_type=[41238],
  ident=[41238],
  smiles=[41238],
  ligand={
    z=[1288419],
    x=[1288419, 12],
    pos=[1288419, 3],
  },
  pocket={
    z=[27393111],
    x=[27393111, 12],
    pos=[27393111, 3],
  },
  pocket_residue={ x=[3498751, 23] },
  (ligand, bond, ligand)={
    edge_index=[2, 2857602],
    edge_attr=[2857602, 4],
  },
  (pocket, bond, pocket)={
    edge_index=[2, 55219856],
    edge_attr=[55219856, 4],
  }
)

In [ ]:
sample = data[1]
node_types, edge_types = sample.metadata()
print(f"Node types: {', '.join([nt for nt in node_types])}")
print(f"Edge types: {', '.join([str(et) for et in edge_types])}")
print(f"Number of ligand heavy atoms: {sample['ligand'].x.size(0)}")
print(f"Number of pocket heavy atoms: {sample['pocket'].x.size(0)}")
print(f"Position of ligand atom indexed 0: {sample['ligand'].pos[0]}")
print(f"Position of pocket atom indexed 0: {sample['pocket'].pos[0]}")

Node types: ligand, pocket, pocket_residue
Edge types: ('ligand', 'bond', 'ligand'), ('pocket', 'bond', 'pocket')
Number of ligand heavy atoms: 26
Number of pocket heavy atoms: 652
Position of ligand atom indexed 0: tensor([ 5.4872, 25.3067, 37.4988])
Position of pocket atom indexed 0: tensor([ 9.5601, 17.7450, 49.1304])


In [ ]:
graf = TransformToComplexGraph()(sample)
graf

HeteroData(
  y=[1],
  docking_score=[1],
  posit_prob=[1],
  predicted_rmsd=[1],
  pocket_sequence='KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVEFCKFGNLSTYLRSFLASRKCIHRDLAARNILLICDFGLA',
  scaffold='C1CCC(CCC2CCC(CC3CCCCC3)C3CCCCC23)CC1',
  activity_type='pIC50',
  ident=[1],
  smiles='Clc1ccc(Nc2nnc(NCc3ccncc3)c3ccccc23)cc1',
  ligand={
    z=[26],
    x=[26, 12],
    pos=[26, 3],
  },
  pocket={
    z=[652],
    x=[652, 12],
    pos=[652, 3],
  },
  pocket_residue={ x=[85, 23] },
  complex={
    x=[678, 12],
    z=[678],
    pos=[678, 3],
  },
  (ligand, bond, ligand)={
    edge_index=[2, 58],
    edge_attr=[58, 4],
  },
  (pocket, bond, pocket)={
    edge_index=[2, 1308],
    edge_attr=[1308, 4],
  },
  (complex, bond, complex)={
    edge_index=[2, 1366],
    edge_attr=[1366, 4],
  }
)

In [ ]:
import networkx as nx
from pyvis.network import Network

def interactive_hetero_graph(data):
    # Initialize PyVis network object with remote CDN resources to avoid permission issues
    net = Network(notebook=True, width="1000px", height="700px", directed=False, cdn_resources="remote")

    # Loop over each node type in HeteroData
    for node_type in data.node_types:
        node_indices = data[node_type].num_nodes
        for i in range(node_indices):
            node_id = f"{node_type}_{i}"
            # Color nodes differently based on their type
            color = 'blue' if node_type == 'protein' else 'green' if node_type == 'ligand' else 'gray'
            net.add_node(node_id, label=node_id, color=color)

    # Loop over each edge type in HeteroData
    for edge_type in data.edge_types:
        edge_index = data[edge_type].edge_index
        source_node_type, relation_type, target_node_type = edge_type

        # Adding edges to PyVis graph
        for i in range(edge_index.size(1)):  # edge_index is (2, num_edges)
            source = edge_index[0, i].item()
            target = edge_index[1, i].item()

            source_id = f"{source_node_type}_{source}"
            target_id = f"{target_node_type}_{target}"

            net.add_edge(source_id, target_id, label=relation_type)

    # Display the interactive graph
    return net.show("hetero_graph.html")

# Usage: assuming your HeteroData object is loaded as 'data'
interactive_hetero_graph(sample)


hetero_graph.html


# Match PyG data to the ChemEMBL activity_ids

In [4]:
_DATA = Path.cwd() / "data"
_DATA

PosixPath('/home/fatemeh/thesis/kinodata-3D-affinity-prediction/data')

In [9]:
from rdkit.Chem import PandasTools

raw_dir = _DATA / "raw"
file_name = "kinodata_docked_v2.sdf.gz",
raw_fp = f"{_DATA}/raw/kinodata_docked_v2.sdf.gz"

raw_data_df = PandasTools.LoadSDF(
    raw_fp,
    smilesName="compound_structures.canonical_smiles",
    molColName="molecule",
    embedProps=True,
    removeHs=True,
)

In [10]:
raw_data_df.head()

,assays.chembl_id,target_dictionary.chembl_id,molecule_dictionary.chembl_id,molecule_dictionary.max_phase,activities.standard_type,activities.standard_value,activities.standard_units,compound_structures.canonical_smiles,compound_structures.standard_inchi,component_sequences.sequence,...,docking.posit_probability,docking.duration,similar.klifs_structure_id,similar.ligand_pdb,similar.complex_pdb,similar.chain,similar.fp_similarity,docking.predicted_rmsd,ID,molecule
0,CHEMBL847682,CHEMBL4128,CHEMBL69638,0,pIC50,5.468521,nM,Nc1ncnc2c1c(-c1cccc(Oc3ccccc3)c1)cn2C1CCCC1,InChI=1S/C23H22N4O/c24-22-21-20(14-27(17-8-4-5...,MDSLASLVLCGVSLLLSGTVEGAMDLILINSLPLVSDAETSLTCIA...,...,0.24,62.318077,5553.0,QQ1,2WQB,A,0.298246,4.790116,32336,<rdkit.Chem.rdchem.Mol object at 0x7fd0526eb1b0>
1,CHEMBL674643,CHEMBL203,CHEMBL306988,0,pIC50,3.30103,nM,CC(=C(C#N)C#N)c1ccc(NC(=O)CCC(=O)[O-])cc1,InChI=1S/C15H13N3O3/c1-10(12(8-16)9-17)11-2-4-...,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...,...,0.05,83.548552,10374.0,FZP,6D8E,A,0.232877,6.358418,32770,<rdkit.Chem.rdchem.Mol object at 0x7fd0526eb140>
2,CHEMBL812621,CHEMBL279,CHEMBL419526,0,pIC50,5.725842,nM,c1ccc(-c2cnc(Nc3ccccn3)o2)cc1,InChI=1S/C14H11N3O/c1-2-6-11(7-3-1)12-10-16-14...,MQSKVLLAVALWLCVETRAASVGLPSVSLDLPRLSIQKDILTIKAN...,...,0.05,513.319727,7065.0,6NC,5JT2,A,0.142857,5.895421,33033,<rdkit.Chem.rdchem.Mol object at 0x7fd0526eb220>
3,CHEMBL664848,CHEMBL3142,CHEMBL104450,0,pIC50,6.455932,nM,O=c1cc(-c2cccc(-c3ccc(O)cc3)c2)sc(N2CCOCC2)c1,InChI=1S/C21H19NO3S/c23-18-6-4-15(5-7-18)16-2-...,MAGSGAGVRCSLLRLQETLSAADRCGAALAGHQLIRGLGQECVLSS...,...,0.18,71.917884,2316.0,07U,3TXO,A,0.163636,4.018659,33375,<rdkit.Chem.rdchem.Mol object at 0x7fd0526eb290>
4,CHEMBL812621,CHEMBL279,CHEMBL330621,0,pIC50,7.721246,nM,Cc1ccc(Nc2ncc(-c3ccccc3)s2)nc1,InChI=1S/C15H13N3S/c1-11-7-8-14(16-9-11)18-15-...,MQSKVLLAVALWLCVETRAASVGLPSVSLDLPRLSIQKDILTIKAN...,...,0.18,71.689741,5323.0,AAZ,1Y6A,A,0.4,4.664443,34276,<rdkit.Chem.rdchem.Mol object at 0x7fd0526eb300>


In [13]:
raw_data_df.columns

Index(['assays.chembl_id', 'target_dictionary.chembl_id',
       'molecule_dictionary.chembl_id', 'molecule_dictionary.max_phase',
       'activities.standard_type', 'activities.standard_value',
       'activities.standard_units', 'compound_structures.canonical_smiles',
       'compound_structures.standard_inchi', 'component_sequences.sequence',
       'assays.confidence_score', 'docs.chembl_id', 'docs.year',
       'docs.authors', 'UniprotID', 'docking.chemgauss_score',
       'docking.posit_probability', 'docking.duration',
       'similar.klifs_structure_id', 'similar.ligand_pdb',
       'similar.complex_pdb', 'similar.chain', 'similar.fp_similarity',
       'docking.predicted_rmsd', 'ID', 'molecule'],
      dtype='object')

Save a mapping of index to activity id (named as ID)

In [ ]:
ident_activityID_map = raw_data_df['ID'].to_dict()
with open("data/probing/ident_activityID_map.json", "w") as f:
    json.dump(ident_activityID_map, f)

In [ ]:
with open("data/probing/ident_activityID_map.json", "r") as f:
    ident_activityID_map = json.load(f)
ident_activityID_map

{'0': '32336',
 '1': '32770',
 '2': '33033',
 '3': '33375',
 '4': '34276',
 '5': '35297',
 '6': '35302',
 '7': '35519',
 '8': '36519',
 '9': '37861',
 '10': '39110',
 '11': '40257',
 '12': '40607',
 '13': '40689',
 '14': '40692',
 '15': '40693',
 '16': '40694',
 '17': '41222',
 '18': '41866',
 '19': '41979',
 '20': '41980',
 '21': '41983',
 '22': '41985',
 '23': '41986',
 '24': '41987',
 '25': '42683',
 '26': '42689',
 '27': '43147',
 '28': '44685',
 '29': '45078',
 '30': '45079',
 '31': '45962',
 '32': '46268',
 '33': '49408',
 '34': '49411',
 '35': '49413',
 '36': '50202',
 '37': '51439',
 '38': '51440',
 '39': '51883',
 '40': '51885',
 '41': '51886',
 '42': '51887',
 '43': '53296',
 '44': '53297',
 '45': '53762',
 '46': '53764',
 '47': '53967',
 '48': '53974',
 '49': '54991',
 '50': '55137',
 '51': '55138',
 '52': '57444',
 '53': '57679',
 '54': '58605',
 '55': '60020',
 '56': '60799',
 '57': '60800',
 '58': '60801',
 '59': '61302',
 '60': '62027',
 '61': '62143',
 '62': '62147',
 '

In [12]:
ident_index_map = data.ident_index_map()

In [82]:
ident_index_map

{20: 0,
 3225: 1,
 3319: 2,
 3435: 3,
 3495: 4,
 3504: 5,
 4872: 6,
 4873: 7,
 4877: 8,
 4895: 9,
 4896: 10,
 5133: 11,
 5134: 12,
 5150: 13,
 5181: 14,
 5182: 15,
 5183: 16,
 5190: 17,
 5191: 18,
 5193: 19,
 6656: 20,
 7966: 21,
 7978: 22,
 7989: 23,
 7990: 24,
 10272: 25,
 10274: 26,
 52853: 27,
 59070: 28,
 10662: 29,
 10664: 30,
 10688: 31,
 16848: 32,
 16849: 33,
 16850: 34,
 16852: 35,
 16853: 36,
 16855: 37,
 16856: 38,
 8653: 39,
 8654: 40,
 8655: 41,
 8656: 42,
 8657: 43,
 16892: 44,
 61379: 45,
 106702: 46,
 10: 47,
 23: 48,
 26: 49,
 27: 50,
 28: 51,
 61: 52,
 69: 53,
 70: 54,
 71: 55,
 75: 56,
 76: 57,
 81: 58,
 227: 59,
 502: 60,
 903: 61,
 920: 62,
 921: 63,
 946: 64,
 953: 65,
 1074: 66,
 1075: 67,
 1086: 68,
 1097: 69,
 1125: 70,
 1134: 71,
 1150: 72,
 1174: 73,
 1175: 74,
 1571: 75,
 2005: 76,
 2010: 77,
 2038: 78,
 2044: 79,
 2047: 80,
 2050: 81,
 2053: 82,
 2057: 83,
 2058: 84,
 2064: 85,
 2065: 86,
 2066: 87,
 2071: 88,
 2076: 89,
 2083: 90,
 2084: 91,
 2085: 92,
 2

Create a mapping of ident to activity.activity_id (which is named ID)

In [91]:
print(ident_activityID_map.keys())

dict_keys([20, 3225, 3319, 3435, 3495, 3504, 4872, 4873, 4877, 4895, 4896, 5133, 5134, 5150, 5181, 5182, 5183, 5190, 5191, 5193, 6656, 7966, 7978, 7989, 7990, 10272, 10274, 52853, 59070, 10662, 10664, 10688, 16848, 16849, 16850, 16852, 16853, 16855, 16856, 8653, 8654, 8655, 8656, 8657, 16892, 61379, 106702, 10, 23, 26, 27, 28, 61, 69, 70, 71, 75, 76, 81, 227, 502, 903, 920, 921, 946, 953, 1074, 1075, 1086, 1097, 1125, 1134, 1150, 1174, 1175, 1571, 2005, 2010, 2038, 2044, 2047, 2050, 2053, 2057, 2058, 2064, 2065, 2066, 2071, 2076, 2083, 2084, 2085, 2087, 2099, 2107, 2493, 2494, 2501, 2510, 2529, 2540, 2557, 2558, 2559, 2794, 2833, 2840, 2841, 2871, 2873, 2892, 2918, 3020, 3037, 3041, 3047, 3054, 3060, 3082, 3083, 3094, 3415, 3426, 3632, 3641, 3653, 3664, 3685, 3857, 3858, 4123, 4126, 4303, 4304, 4305, 4307, 4308, 4958, 4964, 4965, 5559, 5560, 5561, 5562, 5563, 5564, 5565, 5566, 5567, 5568, 7003, 9440, 9441, 9463, 9467, 9469, 9470, 9471, 9472, 9487, 9491, 11776, 12161, 12162, 12163, 1216

In [ ]:
index_activityID_map = {}
for ident, idx in ident_index_map.items():
    index_activityID_map[idx] = ident_activityID_map[ident]
with open("data/probing/index_activity_map.json", "w") as f:
    json.dump(index_activityID_map, f)

index_activityID_map

{0: '32336',
 1: '32770',
 2: '33033',
 3: '33375',
 4: '34276',
 5: '35297',
 6: '35302',
 7: '35519',
 8: '36519',
 9: '37861',
 10: '39110',
 11: '40257',
 12: '40607',
 13: '40689',
 14: '40692',
 15: '40693',
 16: '40694',
 17: '41222',
 18: '41866',
 19: '41979',
 20: '41980',
 21: '41983',
 22: '41985',
 23: '41986',
 24: '41987',
 25: '42683',
 26: '42689',
 27: '43147',
 28: '44685',
 29: '45078',
 30: '45079',
 31: '45962',
 32: '46268',
 33: '49408',
 34: '49411',
 35: '49413',
 36: '50202',
 37: '51439',
 38: '51440',
 39: '51883',
 40: '51885',
 41: '51886',
 42: '51887',
 43: '53296',
 44: '53297',
 45: '53762',
 46: '53764',
 47: '53967',
 48: '53974',
 49: '54991',
 50: '55137',
 51: '55138',
 52: '57444',
 53: '57679',
 54: '58605',
 55: '60020',
 56: '60799',
 57: '60800',
 58: '60801',
 59: '61302',
 60: '62027',
 61: '62143',
 62: '62147',
 63: '62149',
 64: '62715',
 65: '62721',
 66: '64049',
 67: '66108',
 68: '66112',
 69: '66113',
 70: '66857',
 71: '67197',
 7

In [38]:
print(os.getcwd())
templates_dir = "../kinodata-3D/data/templates.csv"
ref_dir = "data/probing/human_kinases_and_chembl_targets.chembl_33.csv"
activity_100 = "data/probing/activities-chembl33-sample100_v0.5.csv"
chembl_activity = "data/probing/activities-chembl33_v0.5.csv"
egfr_activity = "data/probing/EGFR-activities-chembl33.csv"
reference_df = pd.read_csv(ref_dir)
egfr_activity = pd.read_csv(egfr_activity)
activity_100 = pd.read_csv(activity_100)
chembl_activity = pd.read_csv(chembl_activity)

/home/fatemeh/thesis/kinodata-3D-affinity-prediction


In [23]:
data[0]

HeteroData(
  y=[1],
  docking_score=[1],
  posit_prob=[1],
  predicted_rmsd=[1],
  pocket_sequence='KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVEFCKFGNLSTYLRSFLASRKCIHRDLAARNILLICDFGLA',
  scaffold='C1CCC(CC2CCC(CC3CCCC4CCCCC43)CC2)CC1',
  activity_type='pIC50',
  ident=[1],
  smiles='c1ccc(Oc2ccc(Nc3ncnc4ccccc34)cc2)cc1',
  ligand={
    z=[24],
    x=[24, 12],
    pos=[24, 3],
  },
  pocket={
    z=[652],
    x=[652, 12],
    pos=[652, 3],
  },
  pocket_residue={ x=[85, 23] },
  complex={
    x=[676, 12],
    z=[676],
    pos=[676, 3],
  },
  (ligand, bond, ligand)={
    edge_index=[2, 54],
    edge_attr=[54, 4],
  },
  (pocket, bond, pocket)={
    edge_index=[2, 1308],
    edge_attr=[1308, 4],
  },
  (complex, bond, complex)={
    edge_index=[2, 1362],
    edge_attr=[1362, 4],
  }
)

In [39]:
chembl_activity.head()

,Unnamed: 0,activities.activity_id,assays.chembl_id,target_dictionary.chembl_id,molecule_dictionary.chembl_id,molecule_dictionary.max_phase,activities.standard_type,activities.standard_value,activities.standard_units,compound_structures.canonical_smiles,compound_structures.standard_inchi,component_sequences.sequence,assays.confidence_score,docs.chembl_id,docs.year,docs.authors,UniprotID
0,96698,16291323,CHEMBL3705523,CHEMBL2973,CHEMBL3666724,NaN,pIC50,14.096910,nM,CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(O...,InChI=1S/C31H33N7O3/c1-2-4-29(40)33-22-6-3-5-2...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116
1,94326,16264754,CHEMBL3705523,CHEMBL2973,CHEMBL3666728,NaN,pIC50,14.000000,nM,CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(O...,InChI=1S/C34H40N8O3/c1-5-7-32(43)36-24-9-6-8-2...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116
2,98119,16306943,CHEMBL3705523,CHEMBL2973,CHEMBL1968705,NaN,pIC50,14.000000,nM,CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(O...,InChI=1S/C31H33N7O2/c1-2-6-29(39)33-23-8-5-7-2...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116
3,101161,16340050,CHEMBL3705523,CHEMBL2973,CHEMBL1997433,NaN,pIC50,13.958607,nM,CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(O...,InChI=1S/C28H28N6O3/c1-3-5-26(35)30-20-7-4-6-1...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116
4,101541,16344107,CHEMBL3705523,CHEMBL2973,CHEMBL3666722,NaN,pIC50,13.920819,nM,CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(O...,InChI=1S/C32H36N8O2/c1-3-5-30(41)34-24-7-4-6-2...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116


In [10]:
egfr_activity.head()

,Unnamed: 0.1,Unnamed: 0,activities.activity_id,assays.chembl_id,target_dictionary.chembl_id,molecule_dictionary.chembl_id,molecule_dictionary.max_phase,activities.standard_type,activities.standard_value,activities.standard_units,compound_structures.canonical_smiles,compound_structures.standard_inchi,component_sequences.sequence,assays.confidence_score,docs.chembl_id,docs.year,docs.authors,UniprotID
0,57,7654,1044894,CHEMBL683040,CHEMBL203,CHEMBL63786,NaN,pIC50,11.522879,nM,Brc1cccc(Nc2ncnc3cc4ccccc4cc23)c1,InChI=1S/C18H12BrN3/c19-14-6-3-7-15(10-14)22-1...,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...,9,CHEMBL1129035,1996.0,"Rewcastle GW, Palmer BD, Bridges AJ, Showalter...",P00533
1,107,1064,191437,CHEMBL677389,CHEMBL203,CHEMBL35820,NaN,pIC50,11.221849,nM,CCOc1cc2ncnc(Nc3cccc(Br)c3)c2cc1OCC,InChI=1S/C18H18BrN3O2/c1-3-23-16-9-14-15(10-17...,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...,8,CHEMBL1130030,1997.0,"Palmer BD, Trumpp-Kallmeyer S, Fry DW, Nelson ...",P00533
2,108,7949,1082447,CHEMBL680021,CHEMBL203,CHEMBL53711,NaN,pIC50,11.221849,nM,CN(C)c1cc2c(Nc3cccc(Br)c3)ncnc2cn1,InChI=1S/C15H14BrN5/c1-21(2)14-7-12-13(8-17-14...,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...,8,CHEMBL1129564,1996.0,"Rewcastle GW, Palmer BD, Thompson AM, Bridges ...",P00533
3,150,2936,428391,CHEMBL679944,CHEMBL203,CHEMBL66031,NaN,pIC50,11.096910,nM,Brc1cccc(Nc2ncnc3cc4[nH]cnc4cc23)c1,InChI=1S/C15H10BrN5/c16-9-2-1-3-10(4-9)21-15-1...,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...,8,CHEMBL1132555,1999.0,"Showalter HD, Bridges AJ, Zhou H, Sercel AD, M...",P00533
4,151,2647,400160,CHEMBL679944,CHEMBL203,CHEMBL53753,NaN,pIC50,11.096910,nM,CNc1cc2c(Nc3cccc(Br)c3)ncnc2cn1,InChI=1S/C14H12BrN5/c1-16-13-6-11-12(7-17-13)1...,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...,8,CHEMBL1132555,1999.0,"Showalter HD, Bridges AJ, Zhou H, Sercel AD, M...",P00533


In [6]:
activity_100.head(10)

,Unnamed: 0,activities.activity_id,assays.chembl_id,target_dictionary.chembl_id,molecule_dictionary.chembl_id,molecule_dictionary.max_phase,activities.standard_type,activities.standard_value,activities.standard_units,compound_structures.canonical_smiles,compound_structures.standard_inchi,component_sequences.sequence,assays.confidence_score,docs.chembl_id,docs.year,docs.authors,UniprotID
0,96698,16291323,CHEMBL3705523,CHEMBL2973,CHEMBL3666724,NaN,pIC50,14.096910,nM,CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(O...,InChI=1S/C31H33N7O3/c1-2-4-29(40)33-22-6-3-5-2...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116
1,94326,16264754,CHEMBL3705523,CHEMBL2973,CHEMBL3666728,NaN,pIC50,14.000000,nM,CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(O...,InChI=1S/C34H40N8O3/c1-5-7-32(43)36-24-9-6-8-2...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116
2,98119,16306943,CHEMBL3705523,CHEMBL2973,CHEMBL1968705,NaN,pIC50,14.000000,nM,CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(O...,InChI=1S/C31H33N7O2/c1-2-6-29(39)33-23-8-5-7-2...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116
3,101161,16340050,CHEMBL3705523,CHEMBL2973,CHEMBL1997433,NaN,pIC50,13.958607,nM,CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(O...,InChI=1S/C28H28N6O3/c1-3-5-26(35)30-20-7-4-6-1...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116
4,101541,16344107,CHEMBL3705523,CHEMBL2973,CHEMBL3666722,NaN,pIC50,13.920819,nM,CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(O...,InChI=1S/C32H36N8O2/c1-3-5-30(41)34-24-7-4-6-2...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116
5,96324,16287186,CHEMBL3705523,CHEMBL2973,CHEMBL3666721,NaN,pIC50,13.920819,nM,CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(O...,InChI=1S/C32H35N7O2/c1-2-7-30(40)34-24-9-6-8-2...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116
6,99433,16320578,CHEMBL3705523,CHEMBL2973,CHEMBL3666727,NaN,pIC50,13.853872,nM,CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(O...,InChI=1S/C33H38N8O3/c1-4-6-31(42)35-24-8-5-7-2...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116
7,95350,16276310,CHEMBL3705523,CHEMBL2973,CHEMBL3666758,NaN,pIC50,13.619789,nM,CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(O...,InChI=1S/C29H30N6O4/c1-4-6-27(36)31-20-8-5-7-1...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116
8,96706,16291435,CHEMBL3705523,CHEMBL2973,CHEMBL3666756,NaN,pIC50,13.537602,nM,CCCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3ccc...,InChI=1S/C26H24N6O/c1-2-3-11-24(33)28-19-8-6-7...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116
9,100633,16334330,CHEMBL3705523,CHEMBL2973,CHEMBL3666726,NaN,pIC50,13.537602,nM,CCCC(=O)Nc1cccc(-c2nc(Nc3ccc4[nH]ncc4c3)c3cc(O...,InChI=1S/C29H30N6O4/c1-4-6-27(36)31-20-8-5-7-1...,MSRPPPTGKMPGAPETAPGDGAGASRQRKLEALIRDPRSPINVESL...,9,CHEMBL3639077,2014.0,NaN,O75116


### Check PyGs
1. Randomely select 100 Kinodata idents? or it's better to just check the first 100 
What to check for exactly?
    ``` 
    data.smiles == egfr_activity['compound_structures.canonical_smiles']
    data.y == egfr_activity['activities.activity_value'] 
    ```

In [18]:
data

KinodataDocked(41238)

In [31]:
egfr_activity[egfr_activity['compound_structures.canonical_smiles'] == 'Brc1cccc(Nc2ncnc3cc4ccccc4cc23)c1']['compound_structures.canonical_smiles']

0    Brc1cccc(Nc2ncnc3cc4ccccc4cc23)c1
Name: compound_structures.canonical_smiles, dtype: object

In [68]:
type(chembl_activity[chembl_activity['activities.activity_id'] == 2616278 ]['activities.standard_value'].values[0])

numpy.float64

In [43]:
print(min(egfr_activity['activities.activity_id'].values))

32260


In [105]:
import random
# randome_idents = random.sample(list(ident_index_map.keys()), 100)
n = len(data)

def check_info(graph: KinodataDocked,
                activity_ref: pd.DataFrame,
                id: str,
                ref_id_name: str = "activities.activity_id",
                ref_smiles_name: str = "compound_structures.canonical_smiles",
                ref_activity_value: str = "activities.standard_value"
                ):
    id = int(id)
    if id not in activity_ref[ref_id_name].values:
        # print(f"ID {id} not found in the reference activity dataframe")
        return False
    
    # check SMILES
    # ref_smiles = activty_ref[activty_ref[ref_id_name] == id][ref_smiles_name].values[0] # series to value
    # print(f"Ref SMILES: {ref_smiles}")
    # print(f"Graph SMILES: {graph.smiles}")
    # assert graph.smiles == ref_smiles, f"SMILES do not match for Kinodata {graph.ident}"

    # check activity_id
    try:
        ref_activity = activity_ref[activity_ref[ref_id_name] == id][ref_activity_value].values[0]
    except IndexError:
        print(f"activity not found in the reference activity dataframe")
        return False
    # assert graph.y.item() == ref_activity, f"Activity ID do not match for Kinodata {graph.ident}"
    print(f"PyG activity: {graph.y.item():.2f}, chEMBL activity: {ref_activity:.2f}")
    if graph.y.item() != ref_activity:
        return False


    return True

# Check k random graphs
k = 10
not_found = []
found = []
for i in random.sample(range(n+1), k):
# for i in range(10):
    graph = data[i]
    # activity_id = ident_activityID_map[graph.ident.item()]
    if not check_info(graph, activity_ref=chembl_activity, id=index_activityID_map[i]):
        not_found.append(index_activityID_map[i])
    else:
        found.append(index_activityID_map[i])

print(f"Found {len(found)} IDs: {found}")
print(f"Not found {len(not_found)} IDs: {not_found}")

PyG activity: 8.26, chEMBL activity: 3.82
PyG activity: 8.84, chEMBL activity: 5.64
PyG activity: 7.77, chEMBL activity: 7.13
PyG activity: 3.12, chEMBL activity: 7.41
PyG activity: 5.89, chEMBL activity: 6.05
PyG activity: 6.12, chEMBL activity: 8.15
PyG activity: 5.94, chEMBL activity: 5.81
PyG activity: 4.54, chEMBL activity: 5.89
PyG activity: 6.58, chEMBL activity: 8.96
PyG activity: 7.89, chEMBL activity: 7.30
Found 0 IDs: []
Not found 10 IDs: ['459505', '3131720', '189082', '2921949', '2159822', '1149523', '2167616', '7576481', '12177074', '1467281']


In [ ]:
print(data[0].ident.item())
print(data[0])

20
HeteroData(
  y=[1],
  docking_score=[1],
  posit_prob=[1],
  predicted_rmsd=[1],
  pocket_sequence='KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVEFCKFGNLSTYLRSFLASRKCIHRDLAARNILLICDFGLA',
  scaffold='C1CCC(CC2CCC(CC3CCCC4CCCCC43)CC2)CC1',
  activity_type='pIC50',
  ident=[1],
  smiles='c1ccc(Oc2ccc(Nc3ncnc4ccccc34)cc2)cc1',
  ligand={
    z=[24],
    x=[24, 12],
    pos=[24, 3],
  },
  pocket={
    z=[652],
    x=[652, 12],
    pos=[652, 3],
  },
  pocket_residue={ x=[85, 23] },
  complex={
    x=[676, 12],
    z=[676],
    pos=[676, 3],
  },
  (ligand, bond, ligand)={
    edge_index=[2, 54],
    edge_attr=[54, 4],
  },
  (pocket, bond, pocket)={
    edge_index=[2, 1308],
    edge_attr=[1308, 4],
  },
  (complex, bond, complex)={
    edge_index=[2, 1362],
    edge_attr=[1362, 4],
  }
)


# Load

In [12]:
mini_data = torch.load("data/probing/100subset_data.pt", weights_only=False)

In [ ]:
mini_loader = DataLoader(
        mini_data,
        batch_size=20,
    )

### trying even smaller

In [ ]:
nini_data = mini_data[:10]
torch.save(nini_data, "data/probing/10subset_data.pt")

In [3]:
nini_data  = torch.load("data/probing/10subset_data.pt", weights_only=False)
nini_loader = DataLoader(
        nini_data,
        batch_size=20,
    )

In [18]:
nini_data[0]

HeteroData(
  y=[1],
  docking_score=[1],
  posit_prob=[1],
  predicted_rmsd=[1],
  pocket_sequence='KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVEFCKFGNLSTYLRSFLASRKCIHRDLAARNILLICDFGLA',
  scaffold='C1CCC(CC2CCC(CC3CCCC4CCCCC43)CC2)CC1',
  activity_type='pIC50',
  ident=[1],
  smiles='c1ccc(Oc2ccc(Nc3ncnc4ccccc34)cc2)cc1',
  ligand={
    z=[24],
    x=[24, 12],
    pos=[24, 3],
  },
  pocket={
    z=[652],
    x=[652, 12],
    pos=[652, 3],
  },
  pocket_residue={ x=[85, 23] },
  complex={
    x=[676, 12],
    z=[676],
    pos=[676, 3],
  },
  (ligand, bond, ligand)={
    edge_index=[2, 54],
    edge_attr=[54, 4],
  },
  (pocket, bond, pocket)={
    edge_index=[2, 1308],
    edge_attr=[1308, 4],
  },
  (complex, bond, complex)={
    edge_index=[2, 1362],
    edge_attr=[1362, 4],
  }
)

In [5]:
with open("data/probing/10_ident_to_idx.json", "w") as f:
    json.dump(nini_data.ident_index_map(), f)

### trying one graph

In [5]:
a_complex = torch.load("data/probing/a_complex.pt", weights_only=False)

Synthesized batch tensor

In [5]:
a_complex[NodeType.Complex].batch = torch.zeros(a_complex[NodeType.Complex].x.shape[0]).type(torch.int64)
a_complex

HeteroData(
  y=[1],
  docking_score=[1],
  posit_prob=[1],
  predicted_rmsd=[1],
  pocket_sequence='KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVEFCKFGNLSTYLRSFLASRKCIHRDLAARNILLICDFGLA',
  scaffold='C1CCC(CC2CCC(CC3CCCC4CCCCC43)CC2)CC1',
  activity_type='pIC50',
  ident=[1],
  smiles='c1ccc(Oc2ccc(Nc3ncnc4ccccc34)cc2)cc1',
  ligand={
    z=[24],
    x=[24, 12],
    pos=[24, 3],
  },
  pocket={
    z=[652],
    x=[652, 12],
    pos=[652, 3],
  },
  pocket_residue={ x=[85, 23] },
  complex={
    x=[676, 12],
    z=[676],
    pos=[676, 3],
    batch=[676],
  },
  (ligand, bond, ligand)={
    edge_index=[2, 54],
    edge_attr=[54, 4],
  },
  (pocket, bond, pocket)={
    edge_index=[2, 1308],
    edge_attr=[1308, 4],
  },
  (complex, bond, complex)={
    edge_index=[2, 1362],
    edge_attr=[1362, 4],
  }
)

Data module, but crashes

In [5]:
data_module = make_kinodata_module(
    cfg.get("data", "training").update(
        dict(
            batch_size=32,
            split_type="scaffold-k-fold",
            filter_rmsd_max_value=2.0,
            split_index=0,
        )
    ),
    transforms=[TransformToComplexGraph(remove_heterogeneous_representation=False)],
)

: 

# Loading the model

In [6]:
print(Path.cwd())

/home/fatemeh/thesis/kinodata-3D-affinity-prediction


In [6]:
model_dir = Path("models")
# print(model_dir)
assert model_dir.exists()

def path_to_model(rmsd_threshold: int, split_type: str, split_fold: int, model_type: str) -> Path:
    p = model_dir / f"rmsd_cutoff_{rmsd_threshold}" / split_type / str(split_fold) / model_type
    if not p.exists():
        p.mkdir(parents=True)
    return p

cgnn_3d_path = path_to_model(rmsd_threshold=2, split_type="scaffold-k-fold", split_fold=0, model_type="CGNN-3D")
cgnn_3d_ckpt = list(cgnn_3d_path.glob("**/*.ckpt"))[0]
# print(model_ckpt)
cgnn_3d_config = cgnn_3d_path / "config.json"
# print(model_config)

In [7]:
def load_wandb_config(
    config_file: Path
) -> dict[str, Any]:
    with open(config_file, "r") as f_config:
        config = json.load(f_config)
    config = {str(key): value["value"] for key, value in config.items()}
    return config

In [8]:
config = cfg.Config(load_wandb_config(cgnn_3d_config))
cgnn_3d = make_complex_transformer(config)

In [9]:
model_cls = {
    # "DTI": make_dti_baseline,
    # "CGNN": make_complex_transformer,
    "CGNN-3D": make_complex_transformer
}

def load_from_checkpoint(model: RegressionModel, model_ckpt: str) -> RegressionModel:
    ckp = torch.load(model_ckpt, map_location="cpu")
    assert isinstance(model, RegressionModel)
    model.load_state_dict(ckp["state_dict"])
    return model

In [10]:
cgnn_3d_loaded = load_from_checkpoint(model = cgnn_3d, model_ckpt=cgnn_3d_ckpt)
cgnn_3d_loaded.train(False)

ComplexTransformer(
  (criterion): MSELoss()
  (act): SiLU()
  (interaction_module): CombinedInteractions(
    (interactions): ModuleList(
      (0): CovalentInteractions(
        (act): SiLU()
        (lin): Linear(in_features=4, out_features=256, bias=True)
      )
      (1): StructuralInteractions(
        (act): SiLU()
        (distance_embedding): GaussianDistEmbedding()
        (lin): Linear(in_features=256, out_features=256, bias=False)
      )
    )
    (act): SiLU()
  )
  (atomic_num_embedding): Embedding(100, 256)
  (lin_atom_features): Linear(in_features=12, out_features=256, bias=True)
  (attention_blocks): ModuleList(
    (0-2): 3 x SPAB(
      (attention): SparseAttention(
        (lin_query): Linear(in_features=256, out_features=256, bias=False)
        (lin_key_value): Linear(in_features=256, out_features=512, bias=False)
        (lin_bias): Linear(in_features=256, out_features=512, bias=False)
        (lin_out): Linear(in_features=256, out_features=256, bias=False)
   

In [12]:
config

Config(lr=0.0001, act=silu, ln1=True, ln2=True, ln3=True, seed=420, optim=adamw, epochs=300, k_fold=5, min_lr=1e-06, dropout=0, dry_run=False, loss_type=mse, lr_factor=0.9, num_heads=4, use_bonds=True, batch_size=42, data_split=None, edge_types=[['ligand', 'intraacts', 'ligand'], ['ligand', 'interacts', 'pocket'], ['pocket', 'interacts', 'ligand']], graph_norm=False, node_types=['complex'], split_type=scaffold-k-fold, accelerator=gpu, lr_patience=8, num_workers=32, split_index=0, weight_decay=3e-06, edge_attr_size=4, need_distances=False, clip_grad_value=None, hidden_channels=256, remove_hydrogen=True, interaction_modes=['covalent', 'structural'], max_num_neighbors=16, add_docking_scores=False, interaction_radius=6, num_attention_blocks=3, num_residue_features=6, add_artificial_decoys=False, filter_rmsd_max_value=2, accumulate_grad_batches=3, early_stopping_patience=24, additional_atom_features=False, perturb_ligand_positions=0, perturb_pocket_positions=0, perturb_complex_positions=0.1

### Predict

In [10]:
import numpy as np
from kinodata.model.regression import RegressionModel, cat_many
from torch_geometric.loader import DataLoader
from pytorch_lightning import Trainer
import pandas as pd


def predict_df(
    model: RegressionModel,
    loader: DataLoader,
    trainer: Trainer | None = None,
    ckpt_path: str | None = "best",
) -> pd.DataFrame:
    if trainer is None:
        trainer = Trainer()
    dict_list = trainer.predict(model, loader, ckpt_path=ckpt_path)
    return pd.DataFrame(
        {key: np.array(value) for key, value in cat_many(dict_list).items()}
    )

Forward pass for only one

In [18]:
a_pred = cgnn_3d_loaded(a_complex)[0][0].item()	
a_pred

6.6271257400512695

Forward pass for nini

In [9]:
nini_dataset = next(iter(nini_loader))
nini_dataset

HeteroDataBatch(
  y=[10],
  docking_score=[10],
  posit_prob=[10],
  predicted_rmsd=[10],
  pocket_sequence=[10],
  scaffold=[10],
  activity_type=[10],
  ident=[10],
  smiles=[10],
  ligand={
    z=[262],
    x=[262, 12],
    pos=[262, 3],
    batch=[262],
    ptr=[11],
  },
  pocket={
    z=[6520],
    x=[6520, 12],
    pos=[6520, 3],
    batch=[6520],
    ptr=[11],
  },
  pocket_residue={
    x=[850, 23],
    batch=[850],
    ptr=[11],
  },
  complex={
    x=[6782, 12],
    z=[6782],
    pos=[6782, 3],
    batch=[6782],
    ptr=[11],
  },
  (ligand, bond, ligand)={
    edge_index=[2, 582],
    edge_attr=[582, 4],
  },
  (pocket, bond, pocket)={
    edge_index=[2, 13080],
    edge_attr=[13080, 4],
  },
  (complex, bond, complex)={
    edge_index=[2, 13662],
    edge_attr=[13662, 4],
  }
)

In [ ]:
nini_dataset

AttributeError: 'HeteroDataBatch' has no attribute 'pocket'

In [24]:
out_graphs, intermediate_node_reprs, intermediate_edge_reprs = cgnn_3d_loaded(nini_dataset)

In [29]:
torch.save(intermediate_edge_reprs, "data/probing/10subset_intermediate_edge_reprs.pt")
torch.save(intermediate_node_reprs, "data/probing/10subset_intermediate_node_reprs.pt")

Forward pass for mini

In [ ]:
# from kinodata.training.predict import predict_df
df_mini_data = predict_df(cgnn_3d, mini_loader, ckpt_path = cgnn_3d_ckpt)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/home/fatemeh/miniconda3/envs/kinodata3d/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
Restoring states from the checkpoint path at models/rmsd_cutoff_2/scaffold-k-fold/0/CGNN-3D/model.ckpt
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v2.2.2. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint models/

Predicting DataLoader 0: 100%|██████████| 5/5 [00:30<00:00,  0.16it/s]


In [ ]:
df_mini_data

,pred,target
0,6.627124,5.565431
1,6.645761,5.327902
2,6.273721,5.853872
3,6.421379,6.214670
4,6.219614,7.335480
...,...,...
95,7.735117,8.958607
96,6.659325,6.121478
97,7.646967,6.856985
98,7.124436,6.357535


In [ ]:
df_mini_data.to_pickle("/data/probing/y_prob_exp_1.pkl")

# Load from here

In [4]:
print(os.getcwd())

/home/fatemeh/thesis/kinodata-3D-affinity-prediction


In [8]:
num_reprs = config.get("num_attention_blocks")
# prob_dir = "data/probing/"
# prob_dir = "data/probing/one_complex/"

Loading for nini

In [11]:
intermediate_node_reprs = torch.load("data/probing/10subset_intermediate_node_reprs.pt", weights_only=False)
intermediate_edge_reprs = torch.load("data/probing/10subset_intermediate_edge_reprs.pt", weights_only=False)

Node

In [12]:
l1_node_reps, l1_batch = intermediate_node_reprs['layer_1']
l2_node_reps, l2_batch = intermediate_node_reprs['layer_2']
l3_node_reps, l3_batch = intermediate_node_reprs['layer_3']
print(l1_node_reps.shape, l1_batch.shape)
print(l2_node_reps.shape, l2_batch.shape)
print(l3_node_reps.shape, l3_batch.shape)

torch.Size([6782, 256]) torch.Size([6782])
torch.Size([6782, 256]) torch.Size([6782])
torch.Size([6782, 256]) torch.Size([6782])


Which complex in the batch each node belongs to

In [19]:
l1_batch.numpy()

array([0, 0, 0, ..., 9, 9, 9])

Embedding of node 0

In [23]:
l1_node_reps[0]

tensor([ 0.0200,  0.7585, -0.3484,  0.3835, -0.3097, -0.2597, -0.6393, -0.5757,
        -0.5918, -0.5617, -0.8123, -0.2459, -0.6008, -0.6400,  0.3363, -0.2215,
         0.4644, -0.9577, -0.0062,  3.0970,  0.4022,  2.0260,  1.5093, -1.1255,
        -0.8451,  1.6359, -1.0697, -0.5283, -0.5975,  2.0577,  1.8443, -0.1801,
        -0.8154, -0.2125, -0.4587, -0.6479, -0.2282, -0.5257,  0.4545,  0.3910,
         1.0702, -0.1959, -0.1889,  0.6674, -0.0415,  1.0018,  0.4644,  0.0127,
        -0.7523, -0.8671,  3.8887, -1.0138, -0.7219,  0.2256, -0.1281,  0.9337,
        -0.5764, -0.1649, -0.3038,  1.9540, -0.2194,  0.6843, -0.2527, -0.2123,
         0.2200,  0.3960, -1.0390, -0.3688, -0.3258,  0.1840, -0.4090, -0.7660,
        -0.3424, -0.6368, -0.7595, -0.5778, -0.5155,  0.7241, -0.9008,  0.0341,
        -0.1755, -0.8062, -0.7101, -0.6342, -0.3076, -0.2067,  0.5957,  0.7663,
        -0.8358,  2.3242, -1.0362, -0.1427, -0.8269,  0.1739,  0.3451, -0.4920,
        -0.6373, -1.0586, -1.3569,  1.74

Edge

In [13]:
l1_edge_reps, l1_edge_index = intermediate_edge_reprs['layer_1']
print(l1_edge_reps.shape, l1_edge_index.shape)

torch.Size([113774, 256]) torch.Size([2, 113774])


In [32]:
# graf_edge_index, graf_edge = subgraph(complex_indices, edge_index = intermediate_edge_reprs[f"layer_{i+1}"][1], edge_attr = intermediate_edge_reprs[f"layer_{i+1}"][0])

- First row: source node
- Second row: destination node

In [21]:
l1_edge_index

tensor([[   0,    0,    0,  ..., 6781, 6781, 6781],
        [   0,    1,    2,  ..., 6778, 6779, 6781]])

### Unbatch the representations

In [14]:
from torch_geometric.utils import unbatch_edge_index, unbatch, subgraph

# separating the batched node and edge representations
unbatched_nodes = unbatch(l1_node_reps, batch=l1_batch)
unbatched_edge_indices = unbatch_edge_index(l1_edge_index, batch=l1_batch)
unbatched_edges = [subgraph(torch.where(l1_batch == i)[0], edge_index = l1_edge_index, edge_attr = l1_edge_reps)[1] for i in range(l1_batch.max().item() + 1)]

print(unbatched_nodes[0].shape)
print(unbatched_edge_indices[0].shape)
print(unbatched_edges[0].shape)

torch.Size([676, 256])
torch.Size([2, 11340])
torch.Size([11340, 256])


Need to unbatch edge reprs as well
Then, extract the ident of that complex, and create another object that holds the hidden representations? or, just record the necessary information needed for the target molecule property?
Could that be the number of hydrogen atoms?

In [ ]:
# To check if the subgraphs inside the batch are correct, through their edge indices
for i, (nodes, edge_indices) in enumerate(zip(unbatched_nodes, unbatched_edge_indices)):
    print(nodes.shape, edge_indices.shape)
    # complex_indices = torch.where(nini_dataset[NodeType.Complex].batch == i)[0] 
    embed_indices = torch.where(l1_batch == i)[0]
    embed_edge_index, embed_edge = subgraph(embed_indices, edge_index = l1_edge_index, edge_attr = l1_edge_reps)
    
    # whether both tensors have the same shape and values:
    # assert torch.equal(edge_indices, embed_edge_index)
    print(torch.equal(edge_indices, embed_edge_index)) # why thet are not the same?
    
    # src_graph = nini_loader.dataset[i]
    # src_graph.probe = {'node': nodes, 'edge_attr': embed_edge, 'edge_index': edge_indices}
    # nini_loader.dataset[i] = src_graph
    

torch.Size([676, 256]) torch.Size([2, 11340])
True
torch.Size([678, 256]) torch.Size([2, 11374])
False
torch.Size([677, 256]) torch.Size([2, 11357])
False
torch.Size([677, 256]) torch.Size([2, 11357])
False
torch.Size([677, 256]) torch.Size([2, 11357])
False
torch.Size([678, 256]) torch.Size([2, 11374])
False
torch.Size([681, 256]) torch.Size([2, 11425])
False
torch.Size([680, 256]) torch.Size([2, 11408])
False
torch.Size([678, 256]) torch.Size([2, 11374])
False
torch.Size([680, 256]) torch.Size([2, 11408])
False


In [16]:
from torch_geometric.utils import subgraph

sample = subgraph(subset=unbatched_nodes[7], edge_index=l1_edge_index, edge_attr=l1_edge_reps)
sample

IndexError: tensors used as indices must be long, int, byte or bool tensors

Match batch

In [70]:
batch_tensor = nini_dataset[NodeType.Complex].batch
batch_tensor

tensor([0, 0, 0,  ..., 9, 9, 9])

In [86]:
complex_graph = nini_dataset[7]
complex_graph

HeteroData(
  y=[1],
  docking_score=[1],
  posit_prob=[1],
  predicted_rmsd=[1],
  pocket_sequence='KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVEFCKFGNLSTYLRSFLASRKCIHRDLAARNILLICDFGLA',
  scaffold='CC(CC1CCC(C2CCCCC2)CC1)C1CCCC1CCC1CCCCC1',
  activity_type='pIC50',
  ident=[1],
  smiles='O=C(Nc1ccc(-c2ccccc2)cc1)c1scnc1CCc1ccncc1',
  ligand={
    z=[28],
    x=[28, 12],
    pos=[28, 3],
  },
  pocket={
    z=[652],
    x=[652, 12],
    pos=[652, 3],
  },
  pocket_residue={ x=[85, 23] },
  complex={
    x=[680, 12],
    z=[680],
    pos=[680, 3],
  },
  (ligand, bond, ligand)={
    edge_index=[2, 62],
    edge_attr=[62, 4],
  },
  (pocket, bond, pocket)={
    edge_index=[2, 1308],
    edge_attr=[1308, 4],
  },
  (complex, bond, complex)={
    edge_index=[2, 1370],
    edge_attr=[1370, 4],
  }
)

In [ ]:
where = torch.where(batch_tensor == l1_batch[7])

In [18]:
from torch_geometric.utils import subgraph

mask = nini_dataset[NodeType.Complex].batch == 7
print(mask.shape)
node_indices = mask.nonzero().squeeze()
graf = subgraph(node_indices, edge_index = l1_edge_index, edge_attr = l1_edge_reps)
graf

torch.Size([6782])


(tensor([[4744, 4744, 4744,  ..., 5423, 5423, 5423],
         [4744, 4745, 4746,  ..., 5421, 5422, 5423]]),
 tensor([[ 1.3687,  0.9872,  1.6187,  ...,  0.5174, -0.8656,  0.0251],
         [-0.9179, -0.9602,  1.2176,  ...,  1.4060, -0.9959, -1.5741],
         [-0.2900,  0.1892,  0.8599,  ...,  0.2710, -0.8691, -0.8715],
         ...,
         [ 0.7387,  0.1890,  0.3719,  ..., -1.1737, -2.8108, -0.3334],
         [ 0.5311,  0.1831,  0.2935,  ..., -0.9659, -2.4069, -0.3310],
         [ 1.2902,  1.0573,  1.5724,  ..., -0.1667, -1.9189, -0.6477]]))

In [24]:
graf[0].shape

torch.Size([2, 11408])

In [ ]:
nini_dataset[NodeType.Complex].batch.numpy()

tensor([0, 0, 0,  ..., 9, 9, 9])

Loading for mini

In [3]:
with open("data/probing/ident_to_idx.json", "r") as f:
    ident_to_idx = json.load(f)

In [ ]:
prob_dir = "data/probing/one_complex/"
edge_indices = []
edge_reprs = []
node_reprs = []


for i in range(num_reprs):
    edge_indices.append(torch.load(prob_dir + f"edge_index_{i}.pt", weights_only=False))
    edge_reprs.append(torch.load(prob_dir + f"edge_repr_{i}.pt", weights_only=False))
    node_reprs.append(torch.load(prob_dir + f"node_repr_{i}.pt", weights_only=False))

print("Shapes:")
print(f"edge_index:{edge_indices[i].shape}, edge:{edge_reprs[i].shape}, node:{node_reprs[i].shape}")

Shapes:
edge_index:torch.Size([2, 229649]), edge:torch.Size([229649, 256]), node:torch.Size([13757, 256])


tensor([[    0,     0,     0,  ..., 13756, 13756, 13756],
        [    0,     1,     2,  ..., 13747, 13749, 13756]])

In [16]:
y_data = pd.read_pickle("data/probing/y_prob_exp_1.pkl")
y_data

,pred,target
0,6.627124,5.565431
1,6.645761,5.327902
2,6.273721,5.853872
3,6.421379,6.214670
4,6.219614,7.335480
...,...,...
95,7.735117,8.958607
96,6.659325,6.121478
97,7.646967,6.856985
98,7.124436,6.357535


# Hook

In [42]:
from kinodata.model.complex_transformer import ComplexTransformer
from torch import Tensor
from typing import Tuple, Dict
from torch_geometric.data import HeteroData

class ProbingModel(ComplexTransformer):
    def __init__(self, original_model, **kwargs) -> None:
        super().__init__(**kwargs)
        self.original_model = original_model
        self.hidden_states: Dict[str, Tuple[str, Tensor]] = {}

    def forward(self, data: HeteroData) -> Tensor:
        node_store = data[NodeType.Complex]
        node_repr = self.initial_embed_nodes(data)
        edge_index, edge_repr = self.initial_embed_edges(data)

        # Clear previous hidden states
        self.hidden_states.clear()
        
        # Capture the initial representations
        self.hidden_states['initial_node_repr'] = ('initial_node_repr', node_repr.clone())
        self.hidden_states['initial_edge_repr'] = ('initial_edge_repr', edge_repr.clone())
        
        for i, (sparse_attention_block, norm) in enumerate(zip(self.attention_blocks, self.norm_layers)):
            # Run the sparse attention block
            node_repr, edge_repr = sparse_attention_block(node_repr, edge_repr, edge_index)

            # Capture node representation after attention block
            self.hidden_states[f'node_repr_after_block_{i}'] = ('node_repr_after_block', node_repr.clone())
            self.hidden_states[f'edge_repr_after_block_{i}'] = ('edge_repr_after_block', edge_repr.clone())
            
            # Apply normalization
            node_repr = norm(node_repr, node_store.batch)

            # Capture node representation after normalization
            self.hidden_states[f'node_repr_after_norm_{i}'] = ('node_repr_after_norm', node_repr.clone())

        # Aggregate the final node representations
        graph_repr = self.aggr(node_repr, node_store.batch)

        # Capture the final graph representation
        self.hidden_states['final_graph_repr'] = ('final_graph_repr', graph_repr.clone())
        
        return self.out(graph_repr)

In [ ]:
def make_probing_model(model: ComplexTransformer, config: cfg.Config) -> ProbingModel:
    cls = partial(model, config)
    return config.init(cls)